In [1]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# PHASE 13 — MEAN TEACHER — CRIC
# ============================================================

PROJECT_ROOT = Path.cwd().parents[1]

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed"
SPLITS_ROOT = DATA_ROOT / "splits"
RESULTS_ROOT = DATA_ROOT / "results"
CHECKPOINT_ROOT = DATA_ROOT / "checkpoints"


# ------------------------------------------------------------
# CRIC paths
# ------------------------------------------------------------

CRIC_CROPS = PROCESSED_ROOT / "cric_crops"
CRIC_MANIFEST = PROCESSED_ROOT / "cric_learning_units.csv"
CRIC_SPLITS = SPLITS_ROOT / "cric_splits.csv"


# ------------------------------------------------------------
# Phase 13 output directories
# ------------------------------------------------------------

PHASE13_RESULTS = RESULTS_ROOT / "phase13"
PHASE13_CHECKPOINTS = CHECKPOINT_ROOT / "phase13"

PHASE13_RESULTS.mkdir(parents=True, exist_ok=True)
PHASE13_CHECKPOINTS.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)


# ------------------------------------------------------------
# Mean Teacher experiment configuration
# ------------------------------------------------------------

BACKBONE = "densenet121"
NUM_CLASSES = 2

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

LABEL_FRACTION = 0.05


# ------------------------------------------------------------
# Mean Teacher parameters
# ------------------------------------------------------------

EMA_DECAY = 0.99
CONSISTENCY_WEIGHT = 1.0


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ------------------------------------------------------------
# Environment summary
# ------------------------------------------------------------

print("PHASE 13 — MEAN TEACHER — CRIC")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("CRIC crops:", CRIC_CROPS)
print("CRIC manifest:", CRIC_MANIFEST)
print("CRIC splits:", CRIC_SPLITS)

print("\nModel configuration:")
print("Backbone:", BACKBONE)
print("Classes:", NUM_CLASSES)
print("Input size: 128x128")
print("Batch size:", BATCH_SIZE)

print("\nOptimization:")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Max epochs:", EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)

print("\nSemi-SL configuration:")
print("Label fraction:", LABEL_FRACTION)
print("EMA decay:", EMA_DECAY)
print("Consistency weight:", CONSISTENCY_WEIGHT)

print("\nReproducibility:")
print("Random seed:", RANDOM_SEED)

print("\nHardware:")
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
            2
        ),
        "GB"
    )
else:
    print("GPU: Not available")

print("\nPyTorch:", torch.__version__)

print("\n=== PHASE 13 MEAN TEACHER — CRIC SETUP COMPLETE ===")

PHASE 13 — MEAN TEACHER — CRIC
Project root: c:\Users\nanda\Documents\cervical-semi-sl
CRIC crops: c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_crops
CRIC manifest: c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_learning_units.csv
CRIC splits: c:\Users\nanda\Documents\cervical-semi-sl\data\splits\cric_splits.csv

Model configuration:
Backbone: densenet121
Classes: 2
Input size: 128x128
Batch size: 32

Optimization:
Learning rate: 0.0001
Weight decay: 1e-05
Max epochs: 50
Early stopping patience: 8

Semi-SL configuration:
Label fraction: 0.05
EMA decay: 0.99
Consistency weight: 1.0

Reproducibility:
Random seed: 42

Hardware:
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 6.0 GB

PyTorch: 2.11.0+cu128

=== PHASE 13 MEAN TEACHER — CRIC SETUP COMPLETE ===


In [2]:
cric_manifest = pd.read_csv(CRIC_MANIFEST)
cric_splits = pd.read_csv(CRIC_SPLITS)

print("CRIC manifest shape:", cric_manifest.shape)
print("CRIC split shape:", cric_splits.shape)

print("\nSplit counts:")
print(cric_splits["split"].value_counts())

print("\nSplit × binary label:")
print(pd.crosstab(cric_splits["split"], cric_splits["binary_label"]))

print("\nColumns:")
print(cric_splits.columns.tolist())

print("\nExample crop:")
example_crop = PROJECT_ROOT / cric_splits.iloc[0]["crop_path"]
print(example_crop)
print("Exists:", example_crop.exists())

CRIC manifest shape: (11534, 9)
CRIC split shape: (11534, 10)

Split counts:
split
train    8274
test     1721
val      1539
Name: count, dtype: int64

Split × binary label:
binary_label     0     1
split                   
test          1108   613
train         4736  3538
val            935   604

Columns:
['dataset', 'image_filename', 'cell_id', 'x_pixel', 'y_pixel', 'class_bethesda', 'binary_label', 'crop_path', 'crop_valid', 'split']

Example crop:
c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_crops\9ae8a4edde40219bad6303cebc672ee4_cell_1.png
Exists: True


In [3]:
train_df = cric_splits[cric_splits["split"] == "train"].copy()
val_df = cric_splits[cric_splits["split"] == "val"].copy()
test_df = cric_splits[cric_splits["split"] == "test"].copy()

labeled_indices = train_df.sample(
    frac=LABEL_FRACTION,
    random_state=RANDOM_SEED
).index

labeled_df = train_df.loc[labeled_indices].copy()
unlabeled_df = train_df.drop(labeled_indices).copy()

print("CRIC 5% MEAN TEACHER POOLS")
print("=" * 60)

print("Full source training set:", len(train_df))
print("Ground-truth labeled set:", len(labeled_df))
print("Unlabeled set:", len(unlabeled_df))
print("Validation set:", len(val_df))
print("Test set:", len(test_df))

print("\nLabeled class distribution:")
print(labeled_df["binary_label"].value_counts().sort_index())

print("\nUnlabeled class distribution:")
print(unlabeled_df["binary_label"].value_counts().sort_index())

print("\nPool integrity:")
print(
    "Labeled + Unlabeled =",
    len(labeled_df) + len(unlabeled_df),
    "| Full train =",
    len(train_df)
)

CRIC 5% MEAN TEACHER POOLS
Full source training set: 8274
Ground-truth labeled set: 414
Unlabeled set: 7860
Validation set: 1539
Test set: 1721

Labeled class distribution:
binary_label
0    249
1    165
Name: count, dtype: int64

Unlabeled class distribution:
binary_label
0    4487
1    3373
Name: count, dtype: int64

Pool integrity:
Labeled + Unlabeled = 8274 | Full train = 8274


In [4]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cervical_ssl.datasets import CervicalDataset, create_dataloader
from src.cervical_ssl.transforms import (
    get_train_transform,
    get_val_transform
)

train_transform = get_train_transform(BACKBONE)
val_transform = get_val_transform(BACKBONE)

labeled_dataset = CervicalDataset(
    labeled_df,
    transform=train_transform,
    project_root=PROJECT_ROOT
)

unlabeled_dataset = CervicalDataset(
    unlabeled_df,
    transform=train_transform,
    project_root=PROJECT_ROOT
)

val_dataset = CervicalDataset(
    val_df,
    transform=val_transform,
    project_root=PROJECT_ROOT
)

test_dataset = CervicalDataset(
    test_df,
    transform=val_transform,
    project_root=PROJECT_ROOT
)

labeled_loader = create_dataloader(
    labeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

unlabeled_loader = create_dataloader(
    unlabeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = create_dataloader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = create_dataloader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("CRIC MEAN TEACHER DATA LOADERS")
print("=" * 60)

print("Labeled samples:", len(labeled_dataset))
print("Unlabeled samples:", len(unlabeled_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("\nBatches:")
print("Labeled:", len(labeled_loader))
print("Unlabeled:", len(unlabeled_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

images, labels = next(iter(labeled_loader))

print("\nFirst labeled batch:")
print("Images:", images.shape)
print("Labels:", labels.shape)
print("Labels:", labels.tolist())

CRIC MEAN TEACHER DATA LOADERS
Labeled samples: 414
Unlabeled samples: 7860
Validation samples: 1539
Test samples: 1721

Batches:
Labeled: 13
Unlabeled: 246
Validation: 49
Test: 54

First labeled batch:
Images: torch.Size([32, 3, 128, 128])
Labels: torch.Size([32])
Labels: [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0]


In [5]:
from src.cervical_ssl.models import create_model


# ============================================================
# MEAN TEACHER — STUDENT AND TEACHER MODELS
# ============================================================

student_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES
).to(DEVICE)

teacher_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES
).to(DEVICE)


# Initialize teacher with exactly the same weights as the student
teacher_model.load_state_dict(
    student_model.state_dict()
)


# Teacher is not trained through backpropagation
for parameter in teacher_model.parameters():
    parameter.requires_grad = False

teacher_model.eval()


# Student is trained normally
student_model.train()


criterion = nn.CrossEntropyLoss()

student_optimizer = torch.optim.AdamW(
    student_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


print("CRIC MEAN TEACHER MODELS")
print("=" * 60)

print("Student model: DenseNet-121")
print("Teacher model: DenseNet-121")

print("Student device:", next(student_model.parameters()).device)
print("Teacher device:", next(teacher_model.parameters()).device)

print(
    "Student parameters:",
    sum(p.numel() for p in student_model.parameters())
)

print(
    "Teacher parameters:",
    sum(p.numel() for p in teacher_model.parameters())
)

print("\nTeacher gradients enabled:",
      any(p.requires_grad for p in teacher_model.parameters()))

print("Student training mode:", student_model.training)
print("Teacher training mode:", teacher_model.training)

print("\nLoss:", criterion.__class__.__name__)
print("Optimizer:", student_optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

print("\nEMA decay:", EMA_DECAY)
print("Consistency weight:", CONSISTENCY_WEIGHT)

print("\n=== STUDENT AND TEACHER INITIALIZATION COMPLETE ===")

CRIC MEAN TEACHER MODELS
Student model: DenseNet-121
Teacher model: DenseNet-121
Student device: cuda:0
Teacher device: cuda:0
Student parameters: 6955906
Teacher parameters: 6955906

Teacher gradients enabled: False
Student training mode: True
Teacher training mode: False

Loss: CrossEntropyLoss
Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 1e-05

EMA decay: 0.99
Consistency weight: 1.0

=== STUDENT AND TEACHER INITIALIZATION COMPLETE ===


In [6]:
import torch.nn.functional as F


# ============================================================
# MEAN TEACHER HELPER FUNCTIONS
# ============================================================

def update_teacher_ema(
    teacher_model,
    student_model,
    ema_decay
):
    """
    Update teacher parameters using an exponential moving average
    of the student parameters.
    """

    with torch.no_grad():
        for teacher_param, student_param in zip(
            teacher_model.parameters(),
            student_model.parameters()
        ):
            teacher_param.data.mul_(ema_decay)
            teacher_param.data.add_(
                (1.0 - ema_decay) * student_param.data
            )


def supervised_loss(
    student_logits,
    labels
):
    """
    Standard supervised classification loss.
    """
    return criterion(student_logits, labels)


def consistency_loss(
    student_logits,
    teacher_logits
):
    """
    Mean squared error between student and teacher
    probability distributions.
    """

    student_probabilities = torch.softmax(
        student_logits,
        dim=1
    )

    teacher_probabilities = torch.softmax(
        teacher_logits,
        dim=1
    )

    return F.mse_loss(
        student_probabilities,
        teacher_probabilities
    )


print("CRIC MEAN TEACHER HELPER FUNCTIONS")
print("=" * 60)

print("EMA update function: ready")
print("Supervised loss function: ready")
print("Consistency loss function: ready")

print("\nEMA decay:", EMA_DECAY)
print("Consistency weight:", CONSISTENCY_WEIGHT)

print("\n=== MEAN TEACHER MECHANICS READY ===")

CRIC MEAN TEACHER HELPER FUNCTIONS
EMA update function: ready
Supervised loss function: ready
Consistency loss function: ready

EMA decay: 0.99
Consistency weight: 1.0

=== MEAN TEACHER MECHANICS READY ===


In [7]:
from PIL import Image
from torch.utils.data import Dataset


# ============================================================
# MEAN TEACHER — TWO-VIEW UNLABELED DATASET
# ============================================================

class MeanTeacherUnlabeledDataset(Dataset):
    """
    Provides two independently augmented views of each
    unlabeled image.

    View 1 -> Student
    View 2 -> Teacher
    """

    def __init__(
        self,
        dataframe,
        transform,
        project_root
    ):
        self.data = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.project_root = Path(project_root)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        image_path = Path(row["crop_path"])

        if not image_path.is_absolute():
            image_path = self.project_root / image_path

        image = Image.open(image_path).convert("RGB")

        student_view = self.transform(image)
        teacher_view = self.transform(image)

        return student_view, teacher_view


mean_teacher_unlabeled_dataset = MeanTeacherUnlabeledDataset(
    dataframe=unlabeled_df,
    transform=train_transform,
    project_root=PROJECT_ROOT
)

mean_teacher_unlabeled_loader = create_dataloader(
    mean_teacher_unlabeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)


student_view, teacher_view = next(
    iter(mean_teacher_unlabeled_loader)
)

print("CRIC MEAN TEACHER TWO-VIEW DATASET")
print("=" * 60)

print("Unlabeled samples:", len(mean_teacher_unlabeled_dataset))
print("Unlabeled batches:", len(mean_teacher_unlabeled_loader))

print("\nStudent view shape:", student_view.shape)
print("Teacher view shape:", teacher_view.shape)

print(
    "\nStudent and teacher views have same shape:",
    student_view.shape == teacher_view.shape
)

print("\n=== TWO-VIEW UNLABELED DATASET READY ===")

CRIC MEAN TEACHER TWO-VIEW DATASET
Unlabeled samples: 7860
Unlabeled batches: 246

Student view shape: torch.Size([32, 3, 128, 128])
Teacher view shape: torch.Size([32, 3, 128, 128])

Student and teacher views have same shape: True

=== TWO-VIEW UNLABELED DATASET READY ===


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# ============================================================
# CELL 8 — MEAN TEACHER TRAINING LOOP
# ============================================================

def evaluate_teacher(
    teacher_model,
    loader,
    device
):
    """
    Evaluate the teacher model on a labeled validation/test set.
    """

    teacher_model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = teacher_model(images)

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities[:, 1].cpu().numpy()
            )

    labels_np = np.array(all_labels)
    predictions_np = np.array(all_predictions)
    probabilities_np = np.array(all_probabilities)

    accuracy = accuracy_score(
        labels_np,
        predictions_np
    )

    precision = precision_score(
        labels_np,
        predictions_np,
        zero_division=0
    )

    sensitivity = recall_score(
        labels_np,
        predictions_np,
        zero_division=0
    )

    macro_f1 = f1_score(
        labels_np,
        predictions_np,
        average="macro",
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        labels_np,
        predictions_np,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    auroc = roc_auc_score(
        labels_np,
        probabilities_np
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "macro_f1": macro_f1,
        "auroc": auroc
    }


# ============================================================
# RESET STUDENT AND TEACHER MODELS
# ============================================================

student_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES
).to(DEVICE)

teacher_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES
).to(DEVICE)

teacher_model.load_state_dict(
    student_model.state_dict()
)

for parameter in teacher_model.parameters():
    parameter.requires_grad = False

student_model.train()
teacher_model.eval()

student_optimizer = torch.optim.AdamW(
    student_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# CHECKPOINT AND HISTORY SETUP
# ============================================================

mean_teacher_checkpoint_dir = (
    PHASE13_CHECKPOINTS
    / "cric_meanteacher_5pct"
)

mean_teacher_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_checkpoint_path = (
    mean_teacher_checkpoint_dir
    / "best_model.pt"
)

history = []

best_val_macro_f1 = -float("inf")
best_epoch = 0
epochs_without_improvement = 0


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

steps_per_epoch = len(labeled_loader)

print("CRIC MEAN TEACHER TRAINING")
print("=" * 60)

print("Labeled samples:", len(labeled_dataset))
print("Unlabeled samples:", len(mean_teacher_unlabeled_dataset))
print("Validation samples:", len(val_dataset))

print("\nTraining steps per epoch:", steps_per_epoch)
print("Labeled batches:", len(labeled_loader))
print("Unlabeled batches:", len(mean_teacher_unlabeled_loader))

print("\nMaximum epochs:", EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("EMA decay:", EMA_DECAY)
print("Consistency weight:", CONSISTENCY_WEIGHT)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

print("\nStarting training...\n")


# ============================================================
# MAIN TRAINING LOOP
# ============================================================

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    student_model.train()
    teacher_model.eval()

    labeled_iterator = iter(labeled_loader)
    unlabeled_iterator = iter(
        mean_teacher_unlabeled_loader
    )

    running_total_loss = 0.0
    running_supervised_loss = 0.0
    running_consistency_loss = 0.0

    running_correct = 0
    running_samples = 0

    # --------------------------------------------------------
    # Exactly one pass through the labeled batches per epoch
    # --------------------------------------------------------

    for step in range(1, steps_per_epoch + 1):

        # ----------------------------------------------------
        # Labeled batch
        # ----------------------------------------------------

        labeled_images, labels = next(
            labeled_iterator
        )

        # ----------------------------------------------------
        # Unlabeled two-view batch
        # ----------------------------------------------------

        try:

            student_images, teacher_images = next(
                unlabeled_iterator
            )

        except StopIteration:

            unlabeled_iterator = iter(
                mean_teacher_unlabeled_loader
            )

            student_images, teacher_images = next(
                unlabeled_iterator
            )

        # ----------------------------------------------------
        # Move tensors to GPU
        # ----------------------------------------------------

        labeled_images = labeled_images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        student_images = student_images.to(
            DEVICE,
            non_blocking=True
        )

        teacher_images = teacher_images.to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Student supervised prediction
        # ----------------------------------------------------

        labeled_logits = student_model(
            labeled_images
        )

        loss_supervised = supervised_loss(
            labeled_logits,
            labels
        )

        # ----------------------------------------------------
        # Student prediction on unlabeled view
        # ----------------------------------------------------

        student_unlabeled_logits = student_model(
            student_images
        )

        # ----------------------------------------------------
        # Teacher prediction on independent view
        # ----------------------------------------------------

        with torch.no_grad():

            teacher_unlabeled_logits = teacher_model(
                teacher_images
            )

        # ----------------------------------------------------
        # Consistency loss
        # ----------------------------------------------------

        loss_consistency = consistency_loss(
            student_unlabeled_logits,
            teacher_unlabeled_logits
        )

        # ----------------------------------------------------
        # Combined Mean Teacher objective
        # ----------------------------------------------------

        loss_total = (
            loss_supervised
            + CONSISTENCY_WEIGHT * loss_consistency
        )

        # ----------------------------------------------------
        # Student update
        # ----------------------------------------------------

        student_optimizer.zero_grad()

        loss_total.backward()

        student_optimizer.step()

        # ----------------------------------------------------
        # Teacher EMA update
        # ----------------------------------------------------

        update_teacher_ema(
            teacher_model=teacher_model,
            student_model=student_model,
            ema_decay=EMA_DECAY
        )

        # ----------------------------------------------------
        # Running statistics
        # ----------------------------------------------------

        running_total_loss += (
            loss_total.item()
        )

        running_supervised_loss += (
            loss_supervised.item()
        )

        running_consistency_loss += (
            loss_consistency.item()
        )

        labeled_predictions = torch.argmax(
            labeled_logits,
            dim=1
        )

        running_correct += (
            labeled_predictions == labels
        ).sum().item()

        running_samples += labels.size(0)

        # ----------------------------------------------------
        # Progress output
        # ----------------------------------------------------

        if (
            step == 1
            or step % 5 == 0
            or step == steps_per_epoch
        ):

            elapsed = time.time() - epoch_start

            print(
                f"Epoch {epoch:02d} | "
                f"Step {step:02d}/{steps_per_epoch} | "
                f"Loss {loss_total.item():.4f} | "
                f"Sup {loss_supervised.item():.4f} | "
                f"Cons {loss_consistency.item():.4f} | "
                f"Time {elapsed:.1f}s"
            )

    # --------------------------------------------------------
    # Training metrics
    # --------------------------------------------------------

    train_total_loss = (
        running_total_loss
        / steps_per_epoch
    )

    train_supervised_loss = (
        running_supervised_loss
        / steps_per_epoch
    )

    train_consistency_loss = (
        running_consistency_loss
        / steps_per_epoch
    )

    train_accuracy = (
        running_correct
        / running_samples
    )

    # --------------------------------------------------------
    # Teacher validation
    # --------------------------------------------------------

    validation_start = time.time()

    val_metrics = evaluate_teacher(
        teacher_model=teacher_model,
        loader=val_loader,
        device=DEVICE
    )

    validation_time = (
        time.time() - validation_start
    )

    epoch_time = (
        time.time() - epoch_start
    )

    # --------------------------------------------------------
    # Save epoch history
    # --------------------------------------------------------

    epoch_record = {
        "epoch": epoch,
        "train_total_loss": train_total_loss,
        "train_supervised_loss": train_supervised_loss,
        "train_consistency_loss": train_consistency_loss,
        "train_accuracy": train_accuracy,
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_sensitivity": val_metrics["sensitivity"],
        "val_specificity": val_metrics["specificity"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_auroc": val_metrics["auroc"],
        "epoch_time": epoch_time,
        "validation_time": validation_time
    }

    history.append(epoch_record)

    # --------------------------------------------------------
    # Best checkpoint based on validation Macro-F1
    # --------------------------------------------------------

    current_val_f1 = val_metrics["macro_f1"]

    if current_val_f1 > best_val_macro_f1:

        best_val_macro_f1 = current_val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": teacher_model.state_dict(),
                "student_state_dict": student_model.state_dict(),
                "optimizer_state_dict": student_optimizer.state_dict(),
                "best_val_macro_f1": best_val_macro_f1,
                "random_seed": RANDOM_SEED,
                "backbone": BACKBONE,
                "label_fraction": LABEL_FRACTION,
                "ema_decay": EMA_DECAY,
                "consistency_weight": CONSISTENCY_WEIGHT
            },
            best_checkpoint_path
        )

        best_marker = " <-- BEST"

    else:

        epochs_without_improvement += 1
        best_marker = ""

    # --------------------------------------------------------
    # Epoch summary
    # --------------------------------------------------------

    print(
        f"\nEpoch {epoch:02d} COMPLETE | "
        f"Train Loss {train_total_loss:.4f} | "
        f"Train Acc {train_accuracy:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Val F1 {val_metrics['macro_f1']:.4f} | "
        f"Val Sens {val_metrics['sensitivity']:.4f} | "
        f"Val Spec {val_metrics['specificity']:.4f} | "
        f"Val AUROC {val_metrics['auroc']:.4f} | "
        f"Epoch Time {epoch_time:.1f}s"
        f"{best_marker}\n"
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            f"Early stopping at epoch {epoch}."
        )

        break


# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(history)

history_csv_path = (
    PHASE13_RESULTS
    / "cric_meanteacher_5pct_training_history.csv"
)

history_json_path = (
    PHASE13_RESULTS
    / "cric_meanteacher_5pct_training_history.json"
)

history_df.to_csv(
    history_csv_path,
    index=False
)

with open(
    history_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "history": history,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "selection_metric": "macro_f1",
            "steps_per_epoch": steps_per_epoch
        },
        f,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 60)
print("MEAN TEACHER TRAINING COMPLETE")
print("=" * 60)

print("Epochs recorded:", len(history))
print("Steps per epoch:", steps_per_epoch)
print("Best epoch:", best_epoch)

print(
    "Best validation Macro-F1:",
    round(best_val_macro_f1, 6)
)

print(
    "\nCheckpoint:",
    best_checkpoint_path
)

print(
    "Checkpoint exists:",
    best_checkpoint_path.exists()
)

print(
    "\nTraining history:",
    history_csv_path
)

print(
    "\n=== CRIC MEAN TEACHER TRAINING READY FOR EVALUATION ==="
)

CRIC MEAN TEACHER TRAINING
Labeled samples: 414
Unlabeled samples: 7860
Validation samples: 1539

Training steps per epoch: 13
Labeled batches: 13
Unlabeled batches: 246

Maximum epochs: 50
Early stopping patience: 8
EMA decay: 0.99
Consistency weight: 1.0
Learning rate: 0.0001
Weight decay: 1e-05

Starting training...

Epoch 01 | Step 01/13 | Loss 0.7763 | Sup 0.7388 | Cons 0.0375 | Time 5.8s
Epoch 01 | Step 05/13 | Loss 0.5690 | Sup 0.5447 | Cons 0.0244 | Time 31.4s
Epoch 01 | Step 10/13 | Loss 0.7628 | Sup 0.7039 | Cons 0.0589 | Time 50.5s
Epoch 01 | Step 13/13 | Loss 0.4497 | Sup 0.3938 | Cons 0.0560 | Time 63.2s

Epoch 01 COMPLETE | Train Loss 0.5992 | Train Acc 0.7005 | Val Acc 0.6400 | Val F1 0.4899 | Val Sens 0.1242 | Val Spec 0.9733 | Val AUROC 0.6605 | Epoch Time 81.6s <-- BEST

Epoch 02 | Step 01/13 | Loss 0.3351 | Sup 0.2654 | Cons 0.0697 | Time 3.3s
Epoch 02 | Step 05/13 | Loss 0.2829 | Sup 0.2453 | Cons 0.0376 | Time 17.6s
Epoch 02 | Step 10/13 | Loss 0.2432 | Sup 0.1904 